In [1]:
import os
import sys
import json
import yaml
from typing import List
from dotenv import load_dotenv
from pydantic import BaseModel

from agent_framework import Agent, tool
from azure.identity import DefaultAzureCredential
from agent_framework.azure import AzureOpenAIResponsesClient
from agent_framework.devui import serve

# Resolve paths relative to project root
_mqa_dir = os.path.abspath(os.path.join("..", "agents", "mqa-agent"))
_project_root = os.path.abspath("..")

sys.path.insert(0, _project_root)
sys.path.insert(0, _mqa_dir)

from prompt_builder import PromptBuilder

_config_path = os.path.join(_mqa_dir, "config.yaml")
_builder = PromptBuilder(_config_path)

load_dotenv()

True

In [2]:
# ---------------------------------------------------------------------------
# Tool 1: get_available_categories
# ---------------------------------------------------------------------------
@tool
def get_available_categories() -> str:
    """Return the list of available query categories and their associated parameters.

    Call this first to understand which categories can be tagged to a user query.
    Each category has a name and a list of dimension parameters.
    """
    categories = []
    for item in _builder.config.get("categories", []):
        if isinstance(item, dict) and item.get("name"):
            categories.append(
                {"name": item["name"], "parameters": item.get("parameters", [])}
            )
    return json.dumps(categories, indent=2)


# ---------------------------------------------------------------------------
# Tool 2: get_parameters_for_categories
# ---------------------------------------------------------------------------
@tool
def get_parameters_for_categories(categories: List[str]) -> str:
    with open(_config_path, "r") as f:
        config = yaml.safe_load(f)
    parameter_dict = {}
    for category in categories:
        if category not in [c["name"] for c in config["categories"]]:
            continue  # In a real implementation, you might want to handle unknown categories
        for c in config["categories"]:
            if c["name"] == category:
                for param in c["parameters"]:
                    parameter_dict[param] = config["parameters"][param]
    return json.dumps(parameter_dict)

In [3]:
# ---------------------------------------------------------------------------
# Agent instructions
# ---------------------------------------------------------------------------
MQA_INSTRUCTIONS = (
    "You are a Multi-Query Agent designed to help expand user queries into multiple sub-queries based on predefined categories and parameters. "
    "Your goal is to identify relevant categories for a given user query, extract associated parameters, and generate sub-queries that can be used to retrieve data from a database.\n\n"
    "Steps to follow:\n"
    "1. Analyze the user query and determine which categories from the provided list are relevant. You can select multiple categories if applicable.\n"
    "2. For each selected category, identify the associated parameters and their possible values from the configuration.\n"
    "3. Generate multiple sub-queries that combine the user query with the selected categories and parameters. Each sub-query should be a valid question that could be asked to a database or search engine.\n\n"
    "Use the following tools to assist you:\n"
    "- get_available_categories: Returns the list of available categories and their parameters.\n"
    "- get_parameters_for_categories: Given a list of categories, returns the associated parameters and their values.\n\n"
    "Make sure to provide clear and concise sub-queries that cover different aspects of the user's original query based on the selected categories and parameters."
    "Return the sub-queries in a structured format as defined by the MQAResponse model."
)

In [4]:
class MQAResponse(BaseModel):
    sub_queries: List[str]


client = AzureOpenAIResponsesClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    deployment_name=os.environ["AZURE_OPENAI_RESPONSES_DEPLOYMENT_NAME"],
    credential=DefaultAzureCredential(),
)

agent = Agent(
    name="mqa",
    client=client,
    instructions=MQA_INSTRUCTIONS,
    tools=[get_available_categories, get_parameters_for_categories],
)

In [5]:
queries = [
    "How has my drug performed over the last 6 months for Drug D1?",
    "How has my drug performed in 2024 for Drug D1?",
    "How is performance split by shipments, overall patients, and N/R patients for Drug D1?",
    "What has changed over the time period to explain the performance for Drug D1?",
    "For Drug D1, What actions should I take to increase market share, shipments, or patients?",
]


def print_bullets(items):
    for item in items:
        print(f"- {item}")
    print("==" * 20)


for query in queries:
    result = await agent.run(query, options={"response_format": MQAResponse})
    print_bullets(result.value.sub_queries)

- What are the monthly performance trends for Drug D1 over the last 6 months?
- How has Drug D1 performed for new patients over the last 6 months?
- How has Drug D1 performed for refill patients over the last 6 months?
- How has Drug D1 performed for unique patients over the last 6 months?
- What have been the total shipments of Drug D1 over the last 6 months?
- What has been the average shipment quantity for Drug D1 over the last 6 months?
- What has been the average turnaround time for Drug D1 shipments over the last 6 months?
- What were the performance trends for Drug D1 in 2024 by month?
- How did Drug D1 perform in 2024 for new patients versus refill patients?
- What was the performance of Drug D1 in 2024 by patient state?
- What were the total shipments for Drug D1 in 2024?
- What was the average shipment quantity for Drug D1 in 2024?
- What was the average turnaround time for shipments of Drug D1 in 2024?
- What is the performance of Drug D1 split by total shipments, average sh

In [25]:
eval(result.to_dict()["messages"][2]["contents"][0]["arguments"])

{'categories': ['prescriptive_actions',
  'comparative_analysis',
  'performance_trends']}